# Data inventory and wide-receiver coverage

This first-pass audit checks the competition files and the overlap needed for the research question. It does not define the final modeling sample or validate the YAC outcome. Download the files separately as described in the README.

The file inventory scans all nine files, including the large game-tracking CSVs. Tracking frames are read in chunks rather than loaded in full.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'src' / 'nfl_bdb').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook from the repository or one of its subdirectories.')
sys.path.insert(0, str(ROOT / 'src'))
from nfl_bdb.data import coverage_report, file_inventory

DATA_DIR = Path(os.environ.get('NFL_DATA_DIR', ROOT / 'data')).expanduser()
print(f'Data directory: {DATA_DIR}')

## Files and schema

Required columns are checked and the complete file headers are displayed. Scanning the three game-tracking files may take several minutes.

In [ ]:
inventory = file_inventory(DATA_DIR)
display(inventory[['file', 'rows', 'columns', 'size_mb']])
for row in inventory.itertuples(index=False):
    print(f'\n{row.file}: {row.fields}')

## Player coverage and data quality

Tracking coverage counts only `entity_type == PLAYER`. Candidate receiving rows require a regular-season game, a completed pass, non-null receiver yards and YAC, and a play not nullified by penalty. This is a **provisional coverage check**, not a confirmed catch definition; review source records and the `target` flag with the outcome lead before analysis. Lateral plays may meet this filter without being receptions. A zero YAC is retained; missing YAC is not treated as zero.

In [ ]:
report = coverage_report(DATA_DIR)
display(report['summary'])
display(report['by_draft_year'])
display(report['diagnostics'])
display(report['traditional_missingness'])
display(report['wr_drill_coverage'])
display(report['candidate_thresholds'])
display(report['game_periods'])
print('Available Combine PLAYER drill types:', ', '.join(report['tracking_drill_types']))

In [ ]:
coverage = report['by_draft_year'].set_index('draft_year')
coverage[['wr_players', 'player_tracking', 'candidate_yac', 'tracking_and_candidate_yac']].plot.bar(figsize=(9, 4))
plt.title('WR data coverage by draft year (candidate outcome)')
plt.xlabel('Draft year')
plt.ylabel('Number of players')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Next review

Investigate duplicate keys and unmatched game IDs before joining analytical tables. Agree on a reception definition, observation window, and minimum reception count before comparing features or models. Report sample loss at each step; do not interpret this inventory as a final result.